# Does Batch Normalisation Make Training Less Fragile?
**Question:** how much does the learning rate matter for a small CNN on Fashion-MNIST, and does batch norm widen the range of learning rates that work?

Built on the class notebook `hyperparameters_lightning.ipynb`. Only the experiment sections (5 onwards) are new.

**Design (one thing changes at a time):**
- Manipulated: `lr` in 5 values (log scale), measured with batch norm ON and OFF.
- Fixed: Adam, batch size 64, dropout 0.25, 10,000 training images, 5 epochs, no early stopping, no augmentation, same architecture.
- Seeds 42, 43, 44 so we can tell real differences from noise.
- Everything is judged on the **validation** set. The test set is never touched.

In [ ]:
import logging
import random
import time
import warnings
from dataclasses import asdict, dataclass, replace
from pathlib import Path
from typing import Optional

import matplotlib.pyplot as plt
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms

# Lightning 2.x is imported as `lightning`; older installs use `pytorch_lightning`.
try:
    import lightning as L
    from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
    from lightning.pytorch.loggers import CSVLogger
    _LIGHTNING_LOGGER = "lightning.pytorch"
except ImportError:
    import pytorch_lightning as L
    from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
    from pytorch_lightning.loggers import CSVLogger
    _LIGHTNING_LOGGER = "pytorch_lightning"

# Keep the output readable during the many short training runs below.
warnings.filterwarnings("ignore", message=".*does not have many workers.*")
warnings.filterwarnings("ignore", message=".*logging interval.*")

print("torch:", torch.__version__)
print("lightning:", L.__version__)
print("GPU available:", torch.cuda.is_available())